**Requirements (install locally):**
```bash
pip install psycopg2-binary SQLAlchemy geoalchemy2 geopandas shapely pyproj fiona rasterio matplotlib flask
```

**Conda environment (optional):**
```bash
conda create -n postgis python=3.10
conda activate postgis
conda install -c conda-forge psycopg2 sqlalchemy geoalchemy2 geopandas shapely pyproj fiona rasterio matplotlib flask
```


In [ ]:
# 1) Connect to PostGIS
from sqlalchemy import create_engine, text
from sqlalchemy.pool import NullPool

PG_USER = "sakdahomhuan"
PG_PASS = "1234"
PG_HOST = "localhost"
PG_PORT = 5432
PG_DB   = "geo377"

engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}",
    poolclass=NullPool, future=True
)

with engine.begin() as conn:
    version = conn.execute(text("SELECT postgis_full_version();")).scalar_one()
version


In [ ]:
# 2) Initialize schema
sql = open("python_17_postgis_setup.sql", "r", encoding="utf-8").read()
with engine.begin() as conn:
    conn.exec_driver_sql(sql)
print("PostGIS workshop schema initialized.")


In [ ]:
# 3) Checks counts and SRID
from pandas import DataFrame
from sqlalchemy import text
with engine.begin() as conn:
    rs = conn.execute(text("""

        SELECT 'hospital' AS tbl, COUNT(*) AS n, MIN(ST_SRID(geom)) AS srid FROM public.cm_hospital_4326

        UNION ALL

        SELECT 'districts' AS tbl, COUNT(*) AS n, MIN(ST_SRID(geom)) AS srid FROM workshop.districts;

    """))
    df = DataFrame(rs.fetchall(), columns=rs.keys())
df


In [ ]:
# 4) Loads data from PostGIS to GeoDataFrame
import geopandas as gpd
from shapely.geometry import Point

gpoi = gpd.read_postgis(
    "SELECT id, name, geom FROM public.cm_hospital_4326",
    engine, geom_col="geom"
)
display(gpoi.head())


In [ ]:
# 5) Spatial query: find POIs within districts
import pandas as pd
from sqlalchemy import text

q1 = text("""

SELECT p.id, p.name,

       ST_AsText(p.geom) AS wkt,

       ST_Within(p.geom, d.geom) AS inside_district

FROM public.cm_hospital_4326 p

JOIN workshop.districts d

  ON p.geom && d.geom AND ST_Intersects(p.geom, d.geom);

""")
with engine.begin() as conn:
    df = pd.DataFrame(conn.execute(q1).fetchall(), columns=["id","name","wkt","inside_district"])
df


In [ ]:
# 6) Spatial query: KNN (K-Nearest Neighbor) search จาก hospital แต่ละแห่ง หา hospital ที่ใกล้ที่สุด 1 แห่ง
q2 = text("""

SELECT p.id, p.name,

       f.id  AS nearest_id,

       ST_Distance(p.geom::geography, f.geom::geography) AS meters

FROM public.cm_hospital_4326 p

CROSS JOIN LATERAL (

  SELECT id, geom FROM public.cm_hospital_4326 f

  WHERE f.id <> p.id

  ORDER BY f.geom <-> p.geom

  LIMIT 1

) f;

""")

with engine.begin() as conn:

    knn = conn.execute(q2).fetchall()

knn[:5]


In [ ]:
# 7) Spatial query: aggregate hospital counts by district
q3 = text("""

SELECT d.name, COUNT(p.*) AS cnt

FROM workshop.districts d

LEFT JOIN public.cm_hospital_4326 p

  ON d.geom && p.geom AND ST_Intersects(d.geom, p.geom)

GROUP BY d.name ORDER BY cnt DESC;

""")

import pandas as pd

with engine.begin() as conn:

    agg = pd.DataFrame(conn.execute(q3).fetchall(), columns=["district","count"])

agg


In [ ]:
# 8) Export query result as GeoJSON
from sqlalchemy import text
q_geojson = text("""

SELECT jsonb_build_object(

  'type','FeatureCollection',

  'features', jsonb_agg(jsonb_build_object(

     'type','Feature',

     'properties', jsonb_build_object('id', id, 'name', name),

     'geometry', ST_AsGeoJSON(geom)::jsonb

  ))

) AS fc

FROM public.cm_hospital_4326;

""")

with engine.begin() as conn:

    fc = conn.execute(q_geojson).scalar_one()

fc


In [ ]:
# show with folium
import folium
m = folium.Map(location=[18.79, 98.98], zoom_start=12)
# folium.GeoJson(fc, name="geojson").add_to(m)
# add marker popup 
for feature in fc['features']:
    coords = feature['geometry']['coordinates'][::-1]  # reverse to (lat, lon)
    name = feature['properties']['name']
    folium.Marker(location=coords, popup=name).add_to(m)

m
#  --- IGNORE ---   